In [1]:
import os
import sqlite3
import pandas as pd

# Configuración de rutas
csv_path = './dataset.csv' # Ajustado a la ruta directa del archivo

if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"No existe el archivo CSV en: {csv_path}")

try:
    # Carga inicial de datos
    df = pd.read_csv(csv_path)
    print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

    # Conexión a la base de datos
    conn = sqlite3.connect('spotify.db')
    cursor = conn.cursor()
    cursor.execute('PRAGMA foreign_keys = ON;') # Activado para mantener integridad

    # 1. Tabla: albums
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS albums (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            album_name TEXT,
            artists TEXT,
            popularity INTEGER
        )
    ''')

    df_albums = df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']].drop_duplicates(subset=['track_id']).fillna('')
    cursor.executemany('INSERT OR REPLACE INTO albums VALUES (?, ?, ?, ?, ?)', list(df_albums.itertuples(index=False, name=None)))
    print(f"✅ Tabla 'albums' lista ({len(df_albums)} registros)")

    # 2. Tabla: genre
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS genre (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            track_genre TEXT
        )
    ''')

    df_genre = df[['track_id', 'track_name', 'track_genre']].drop_duplicates(subset=['track_id']).fillna('')
    cursor.executemany('INSERT OR REPLACE INTO genre VALUES (?, ?, ?)', list(df_genre.itertuples(index=False, name=None)))
    print(f"✅ Tabla 'genre' lista ({len(df_genre)} registros)")

    # 3. Tabla: Audio_Features
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Audio_Features (
            track_id TEXT PRIMARY KEY,
            danceability REAL NOT NULL,
            energy REAL NOT NULL,
            "key" INTEGER,
            loudness REAL,
            mode INTEGER,
            speechiness REAL,
            acousticness REAL,
            instrumentalness REAL,
            liveness REAL,
            valence REAL,
            tempo REAL,
            time_signature INTEGER
        )
    ''')

    cols_features = ['track_id', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 
                     'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
    df_features = df[cols_features].drop_duplicates(subset=['track_id']).fillna(0)
    cursor.executemany('INSERT OR REPLACE INTO Audio_Features VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', list(df_features.itertuples(index=False, name=None)))
    print(f"✅ Tabla 'Audio_Features' lista ({len(df_features)} registros)")

    # 4. Tabla: popularity_analysis (LA NUEVA TABLA SOLICITADA)
    cursor.execute('DROP TABLE IF EXISTS popularity_analysis') # Reiniciamos para asegurar estructura
    cursor.execute('''
        CREATE TABLE popularity_analysis (
            track_id VARCHAR(255) PRIMARY KEY, 
            track_name VARCHAR(255) NOT NULL,  
            artist_id VARCHAR(255),            
            track_genre VARCHAR(255),        
            album_id VARCHAR(255),             
            popularity INT,                    
            danceability DECIMAL(5, 4)         
        )
    ''')

    # Población de la tabla mediante JOINs de las tablas previas
    cursor.execute('''
        INSERT INTO popularity_analysis (
            track_id, 
            track_name, 
            artist_id, 
            track_genre, 
            album_id, 
            popularity, 
            danceability
        )
        SELECT 
            a.track_id,
            a.track_name,
            a.artists AS artist_id,
            g.track_genre,
            a.album_name AS album_id,
            a.popularity,
            af.danceability
        FROM 
            albums a
        JOIN 
            Audio_Features af ON a.track_id = af.track_id
        JOIN 
            genre g ON a.track_id = g.track_id 
        WHERE 
            a.popularity IS NOT NULL 
            AND a.popularity > 0
        ORDER BY 
            a.popularity DESC
    ''')

    conn.commit()
    print(f"✅ Tabla 'popularity_analysis' creada y poblada mediante JOINs")
    
    # Verificación rápida
    cursor.execute("SELECT COUNT(*) FROM popularity_analysis")
    print(f"📊 Registros en tabla de análisis: {cursor.fetchone()[0]}")

    conn.close()
    print("🎉 Proceso finalizado correctamente: spotify.db")

except Exception as e:
    print(f"❌ Error: {e}")

FileNotFoundError: No existe el archivo CSV en: ./dataset.csv

In [ ]:
import os
import sqlite3
import pandas as pd

# Ruta de salida
output_folder = './tablas en csv'
os.makedirs(output_folder, exist_ok=True)

# Conectar a la base de datos
conn = sqlite3.connect('spotify.db')

# Exportar tablas
df_albums = pd.read_sql_query('SELECT * FROM albums', conn)
df_genre = pd.read_sql_query('SELECT * FROM genre', conn)
df_features = pd.read_sql_query('SELECT * FROM Audio_Features', conn)

# Guardar en CSV
df_albums.to_csv(os.path.join(output_folder, 'albums.csv'), index=False, encoding='utf-8')
df_genre.to_csv(os.path.join(output_folder, 'genre.csv'), index=False, encoding='utf-8')
df_features.to_csv(os.path.join(output_folder, 'Audio_Features.csv'), index=False, encoding='utf-8')

conn.close()

print(f"✅ Exportadas 3 tablas en: {output_folder}")

✅ Exportadas 3 tablas en: ./tablas en csv


In [ ]:
import os
import sqlite3
import pandas as pd

# -------------------------
# 1) Cargar dataset original
# -------------------------
csv_path = './dataset.csv/dataset.csv'

if not os.path.isfile(csv_path):
    raise FileNotFoundError(f"No existe el archivo: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

# -------------------------
# 2) Crear / conectar BD
# -------------------------
conn = sqlite3.connect('spotify.db')
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;')

# -------------------------
# 3) Tabla albums
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS albums (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        album_name TEXT,
        artists TEXT,
        popularity INTEGER
    )
''')

df_albums = df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']].drop_duplicates(subset=['track_id']).fillna('')
cursor.executemany('INSERT OR REPLACE INTO albums VALUES (?, ?, ?, ?, ?)', list(df_albums.itertuples(index=False, name=None)))
print(f"✅ Insertados {len(df_albums)} registros en 'albums'")

# -------------------------
# 4) Tabla genre
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS genre (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        track_genre TEXT
    )
''')

df_genre = df[['track_id', 'track_name', 'track_genre']].drop_duplicates(subset=['track_id']).fillna('')
cursor.executemany('INSERT OR REPLACE INTO genre VALUES (?, ?, ?)', list(df_genre.itertuples(index=False, name=None)))
print(f"✅ Insertados {len(df_genre)} registros en 'genre'")

# -------------------------
# 5) Tabla Audio_Features
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Features (
        track_id TEXT PRIMARY KEY,
        danceability REAL NOT NULL,
        energy REAL NOT NULL,
        "key" INTEGER,
        loudness REAL,
        mode INTEGER,
        speechiness REAL,
        acousticness REAL,
        instrumentalness REAL,
        liveness REAL,
        valence REAL,
        tempo REAL,
        time_signature INTEGER
    )
''')

df_features = df[['track_id', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 
                 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']].drop_duplicates(subset=['track_id']).fillna(0)
cursor.executemany('INSERT OR REPLACE INTO Audio_Features VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', list(df_features.itertuples(index=False, name=None)))
print(f"✅ Insertados {len(df_features)} registros en 'Audio_Features'")

# -------------------------
# 6) Tabla popularity_analysis (ESTRUCTURA SOLICITADA)
# -------------------------
cursor.execute('DROP TABLE IF EXISTS popularity_analysis')
cursor.execute('''
    CREATE TABLE popularity_analysis (
        track_id VARCHAR(255) PRIMARY KEY, 
        track_name VARCHAR(255) NOT NULL,  
        artist_id VARCHAR(255),            
        track_genre VARCHAR(255),        
        album_id VARCHAR(255),             
        popularity INT,                    
        danceability DECIMAL(5, 4)         
    )
''')

# Inserción mediante JOIN de las tablas ya normalizadas
cursor.execute('''
    INSERT INTO popularity_analysis
    SELECT 
        a.track_id,
        a.track_name,
        a.artists AS artist_id,
        g.track_genre,
        a.album_name AS album_id,
        a.popularity,
        af.danceability
    FROM 
        albums a
    JOIN 
        Audio_Features af ON a.track_id = af.track_id
    JOIN 
        genre g ON a.track_id = g.track_id 
    WHERE 
        a.popularity IS NOT NULL 
        AND a.popularity > 0
    ORDER BY 
        a.popularity DESC
''')

# Visualización rápida de resultados
cursor.execute('SELECT track_name, track_genre, popularity FROM popularity_analysis LIMIT 15')
results = cursor.fetchall()

print(f"\n{'Track Name':<40} | {'Genre':<20} | {'Popularity'}")
print("-" * 75)
for row in results:
    print(f"{row[0][:40]:<40} | {row[1]:<20} | {row[2]}")

conn.commit()

# -------------------------
# 7) Exportar tablas a CSV
# -------------------------
output_folder = './tablas en csv'
os.makedirs(output_folder, exist_ok=True)

# Exportamos las tablas principales y la nueva tabla de análisis
tablas = ['albums', 'genre', 'Audio_Features', 'popularity_analysis']

for tabla in tablas:
    pd.read_sql_query(f'SELECT * FROM {tabla}', conn).to_csv(
        os.path.join(output_folder, f'{tabla}.csv'),
        index=False,
        encoding='utf-8'
    )

conn.close()
print(f"\n🎉 Proceso completado. CSV exportados en: {output_folder}")

Dataset cargado: 114000 filas, 21 columnas
Columnas disponibles: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
✅ Insertados 89741 álbumes únicos
✅ Insertados 89741 géneros únicos
✅ Insertados 89741 features únicos
🎉 Proceso completado
📁 CSV exportados en: ./tablas en csv


In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

# -------------------------
# 1) Definir rutas del proyecto
# -------------------------
project_dir = Path(r"C:\Users\juanb\Desktop\IronHack\SQL\Proyecto SQL\sql-database")
csv_path = project_dir / "dataset.csv" / "dataset.csv"
db_path = project_dir / "spotify.db"
output_folder = project_dir / "output"

# Crear carpeta output si no existe
output_folder.mkdir(parents=True, exist_ok=True)

# Verificar existencia del CSV
if not csv_path.is_file():
    raise FileNotFoundError(f"No existe el archivo: {csv_path}")

# -------------------------
# 2) Cargar dataset original
# -------------------------
df = pd.read_csv(csv_path)
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

# -------------------------
# 3) Crear / conectar BD
# -------------------------
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;') # Activado para integridad

# -------------------------
# 4) Tabla albums
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS albums (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        album_name TEXT,
        artists TEXT,
        popularity INTEGER
    )
''')

df_albums = (
    df[['track_id', 'track_name', 'album_name', 'artists', 'popularity']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)

cursor.executemany('INSERT OR REPLACE INTO albums VALUES (?, ?, ?, ?, ?)', 
                   list(df_albums.itertuples(index=False, name=None)))
print(f"✅ Tabla 'albums' lista ({len(df_albums)} registros)")

# -------------------------
# 5) Tabla genre
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS genre (
        track_id TEXT PRIMARY KEY,
        track_name TEXT NOT NULL,
        track_genre TEXT
    )
''')

df_genre = (
    df[['track_id', 'track_name', 'track_genre']]
    .drop_duplicates(subset=['track_id'])
    .fillna('')
)

cursor.executemany('INSERT OR REPLACE INTO genre VALUES (?, ?, ?)', 
                   list(df_genre.itertuples(index=False, name=None)))
print(f"✅ Tabla 'genre' lista ({len(df_genre)} registros)")

# -------------------------
# 6) Tabla Audio_Features
# -------------------------
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Audio_Features (
        track_id TEXT PRIMARY KEY,
        danceability REAL NOT NULL,
        energy REAL NOT NULL,
        "key" INTEGER,
        loudness REAL,
        mode INTEGER,
        speechiness REAL,
        acousticness REAL,
        instrumentalness REAL,
        liveness REAL,
        valence REAL,
        tempo REAL,
        time_signature INTEGER
    )
''')

feature_cols = [
    'track_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'time_signature'
]

df_features = (
    df[feature_cols]
    .drop_duplicates(subset=['track_id'])
    .fillna(0)
)

cursor.executemany('INSERT OR REPLACE INTO Audio_Features VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)', 
                   list(df_features.itertuples(index=False, name=None)))
print(f"✅ Tabla 'Audio_Features' lista ({len(df_features)} registros)")

# -------------------------
# 7) Tabla popularity_analysis (NUEVA TABLA SOLICITADA)
# -------------------------
cursor.execute('DROP TABLE IF EXISTS popularity_analysis')
cursor.execute('''
    CREATE TABLE popularity_analysis (
        track_id VARCHAR(255) PRIMARY KEY, 
        track_name VARCHAR(255) NOT NULL,  
        artist_id VARCHAR(255),            
        track_genre VARCHAR(255),        
        album_id VARCHAR(255),             
        popularity INT,                    
        danceability DECIMAL(5, 4)         
    )
''')

# Inserción mediante JOIN de las tablas ya pobladas
cursor.execute('''
    INSERT INTO popularity_analysis
    SELECT 
        a.track_id,
        a.track_name,
        a.artists AS artist_id,
        g.track_genre,
        a.album_name AS album_id,
        a.popularity,
        af.danceability
    FROM 
        albums a
    JOIN 
        Audio_Features af ON a.track_id = af.track_id
    JOIN 
        genre g ON a.track_id = g.track_id 
    WHERE 
        a.popularity IS NOT NULL 
        AND a.popularity > 0
    ORDER BY 
        a.popularity DESC
''')

conn.commit()
print(f"✅ Tabla 'popularity_analysis' creada y poblada")

# -------------------------
# 8) Exportar tablas a CSV
# -------------------------
tablas = {
    'albums': 'albums.csv',
    'genre': 'genre.csv',
    'Audio_Features': 'Audio_Features.csv',
    'popularity_analysis': 'popularity_analysis.csv'
}

for tabla, archivo in tablas.items():
    df_export = pd.read_sql_query(f'SELECT * FROM {tabla}', conn)
    df_export.to_csv(
        output_folder / archivo,
        index=False,
        encoding='utf-8'
    )
    print(f"📁 Exportado: {archivo}")

# -------------------------
# 9) Vista previa rápida de la tabla solicitada
# -------------------------
print("\n--- Vista previa popularity_analysis (Top 10) ---")
print(pd.read_sql_query('SELECT track_name, track_genre, popularity FROM popularity_analysis LIMIT 10', conn))

conn.close()

print("\n🎉 Proceso completado")
print(f"📂 Base de datos en: {db_path}")
print(f"📂 CSV exportados en: {output_folder}")

PermissionError: [WinError 5] Acceso denegado: 'C:\\Users\\juanb'